# 01 — Recruitment matching system architecture

> **Applied Labs** · **Domain**: AG · **Difficulty**: Advanced · **Reading time**: ~35 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/labs/08_recruitment_matching/01_architecture.ipynb)

**Prerequisites**:
- [00_first_principles.ipynb](./00_first_principles.ipynb)

**What you will learn**:
- End-to-end architecture for a recruitment matching system
- How to parse job descriptions into structured requirement sets
- How to extract skills (explicit and inferred) from resumes
- Semantic matching strategies: exact, semantic, partial, none
- Structured rubric evaluation with weighted criteria
- Bias detection with adverse impact ratio and JD language analysis

In [ ]:
# @title Setup — Run this cell first
!pip install -q "openai>=1.0.0" "sentence-transformers>=2.2.0" "pandas>=2.0.0"

import os, json, re, textwrap
from dataclasses import dataclass, field, asdict
from typing import Optional, Dict, List, Tuple
import numpy as np
import pandas as pd

# ── OpenAI client (optional — used for LLM parsing) ──
from openai import OpenAI

client: Optional[OpenAI] = None
api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    client = OpenAI(api_key=api_key)
    print("OpenAI client initialised ✓")
else:
    print("⚠ OPENAI_API_KEY not set — LLM calls will use mock responses")

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

encoder = SentenceTransformer("all-MiniLM-L6-v2")
print("Sentence encoder loaded ✓")
print("Setup complete ✓")

## Section 1 — System architecture

The recruitment matching system transforms unstructured job descriptions and
resumes into structured, comparable representations, then evaluates alignment
across multiple dimensions with bias safeguards.

```
┌─────────────┐    ┌──────────────────┐    ┌─────────────────┐
│   Job Desc   │───▶│  JD Parser &     │───▶│   Structured    │
│   (raw text) │    │  Req. Extractor  │    │   Requirements  │
└─────────────┘    └──────────────────┘    └────────┬────────┘
                                                     │
                                                     ▼
┌─────────────┐    ┌──────────────────┐    ┌─────────────────┐
│   Resume     │───▶│  Resume Parser & │───▶│   Candidate     │
│   (raw text) │    │  Skill Extractor │    │   Profile       │
└─────────────┘    └──────────────────┘    └────────┬────────┘
                                                     │
                          ┌──────────────────────────┘
                          ▼
               ┌──────────────────┐
               │  Semantic Matcher │
               │  (per-requirement │
               │   evidence search)│
               └────────┬─────────┘
                        │
           ┌────────────┼────────────┐
           ▼            ▼            ▼
    ┌────────────┐ ┌──────────┐ ┌────────────┐
    │   Rubric   │ │   Bias   │ │  Interview │
    │  Evaluator │ │  Checker │ │  Question  │
    └─────┬──────┘ └────┬─────┘ │  Generator │
          │              │       └─────┬──────┘
          ▼              ▼             ▼
    ┌─────────────────────────────────────────┐
    │          Candidate Scorecard             │
    │  (scores, evidence, explanation, flags)  │
    └─────────────────────────────────────────┘
```

Each component has a clear interface: structured input → structured output, making
the system testable, auditable, and fair.

In [ ]:
# ── Pipeline stage registry ──

@dataclass
class PipelineStage:
    """Metadata for a pipeline stage."""
    name: str
    input_type: str
    output_type: str
    description: str
    requires_llm: bool = False

PIPELINE: List[PipelineStage] = [
    PipelineStage("JD Parser", "str (raw JD text)",
                  "List[Requirement]", "Extract structured requirements from JD", True),
    PipelineStage("Resume Parser", "str (raw resume text)",
                  "CandidateProfile", "Parse resume into structured profile", True),
    PipelineStage("Semantic Matcher", "List[Requirement] × CandidateProfile",
                  "List[MatchResult]", "Match each requirement to resume evidence", False),
    PipelineStage("Rubric Evaluator", "List[MatchResult]",
                  "Scorecard", "Score candidate on weighted criteria", False),
    PipelineStage("Bias Checker", "List[Scorecard]",
                  "FairnessReport", "Detect adverse impact and language bias", False),
    PipelineStage("Question Generator", "Scorecard",
                  "List[str]", "Generate tailored interview questions", True),
]

print("=" * 70)
print("  RECRUITMENT MATCHING PIPELINE — STAGE REGISTRY")
print("=" * 70)
for i, stage in enumerate(PIPELINE, 1):
    llm_tag = " [LLM]" if stage.requires_llm else ""
    print(f"  {i}. {stage.name}{llm_tag}")
    print(f"     {stage.input_type} → {stage.output_type}")
    print(f"     {stage.description}")
    print()
print(f"✓ {len(PIPELINE)} stages registered — "
      f"{sum(s.requires_llm for s in PIPELINE)} require LLM")

## Section 2 — JD parsing

Job descriptions mix requirements of different types and priorities. Our parser
extracts:

- **Required skills** vs **preferred skills** (must-have vs nice-to-have)
- **Experience level** (years, seniority)
- **Responsibilities** (what the role actually does)
- **Education requirements**

The output is a structured `ParsedJD` that downstream components consume.

In [ ]:
@dataclass
class Requirement:
    """A single extracted requirement from a JD."""
    id: str
    text: str
    category: str        # technical | experience | education | soft-skill
    priority: str        # required | preferred | nice-to-have
    min_years: float = 0.0

@dataclass
class ParsedJD:
    """Structured output from JD parsing."""
    title: str
    requirements: List[Requirement] = field(default_factory=list)
    responsibilities: List[str] = field(default_factory=list)
    level: str = ""

JD_PARSE_PROMPT = """You are an expert technical recruiter. Parse this job description
into structured requirements.

JOB DESCRIPTION:
{jd_text}

Extract ALL requirements. For each, determine:
- category: technical | experience | education | soft-skill
- priority: required | preferred | nice-to-have
- min_years: minimum years of experience (0 if not specified)

Respond in this exact JSON format:
{{
  "title": "<job title>",
  "level": "<junior|mid|senior|lead|principal>",
  "requirements": [
    {{"id": "R1", "text": "<requirement>", "category": "<cat>", "priority": "<pri>", "min_years": <float>}},
    ...
  ],
  "responsibilities": ["<resp1>", "<resp2>", ...]
}}"""

def parse_jd(jd_text: str) -> ParsedJD:
    """Parse a job description into structured requirements."""
    if client:
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": JD_PARSE_PROMPT.format(jd_text=jd_text)}],
            temperature=0.0,
            response_format={"type": "json_object"},
        )
        data = json.loads(resp.choices[0].message.content)
    else:
        # ── Mock: rule-based extraction ──
        reqs: List[Dict] = []
        for i, line in enumerate(jd_text.split("\n"), 1):
            line = line.strip("- •*")
            if not line or len(line) < 5:
                continue
            cat = "technical"
            pri = "required"
            years = 0.0
            if any(w in line.lower() for w in ["prefer", "nice", "bonus", "plus"]):
                pri = "preferred"
            if any(w in line.lower() for w in ["degree", "bachelor", "master", "phd"]):
                cat = "education"
            elif any(w in line.lower() for w in ["communicat", "collaborat", "leader", "team"]):
                cat = "soft-skill"
            yr_match = re.search(r"(\d+)\+?\s*(?:years?|yrs?)", line.lower())
            if yr_match:
                years = float(yr_match.group(1))
                cat = "experience"
            if line.strip():
                reqs.append({"id": f"R{len(reqs)+1}", "text": line.strip(),
                             "category": cat, "priority": pri, "min_years": years})
        data = {"title": "Parsed Role", "level": "senior",
                "requirements": reqs[:12], "responsibilities": []}

    parsed = ParsedJD(
        title=data.get("title", ""),
        level=data.get("level", ""),
        requirements=[Requirement(**r) for r in data.get("requirements", [])],
        responsibilities=data.get("responsibilities", []),
    )
    return parsed

# ── Test with a sample JD ──
sample_jd = """Senior Machine Learning Engineer

We are looking for a Senior ML Engineer to join our AI team.

Requirements:
- 5+ years of Python development experience
- 3+ years of machine learning model development and deployment
- Strong experience with PyTorch or TensorFlow
- Experience with MLOps tools (MLflow, Kubeflow, or similar)
- Proficiency in SQL and data pipeline tools
- Bachelor's degree in Computer Science or related field

Nice to have:
- Experience with large language models and transformer architectures
- Kubernetes and container orchestration experience
- Published research or conference presentations
- Experience leading a team of 2-5 engineers"""

parsed = parse_jd(sample_jd)
print("=" * 60)
print(f"  PARSED JD: {parsed.title}")
print(f"  Level: {parsed.level}")
print("=" * 60)
for req in parsed.requirements:
    tag = f"[{req.priority.upper()}]"
    yrs = f" ({req.min_years:.0f}+ yrs)" if req.min_years > 0 else ""
    print(f"  {req.id} {tag:<12} {req.category:<12} {req.text}{yrs}")
print(f"\n✓ Extracted {len(parsed.requirements)} requirements from JD")

## Section 3 — Resume understanding

Resume parsing extracts a structured candidate profile including:

- **Explicit skills** — directly mentioned technologies and tools
- **Inferred skills** — derived from experience descriptions (e.g.,
  "built microservices" → Docker, REST APIs, distributed systems)
- **Experience** — years, domains, progression
- **Education** — degrees, fields, institutions
- **Achievements** — quantified accomplishments

In [ ]:
@dataclass
class ExtractedSkill:
    """Skill extracted from a resume."""
    name: str
    source: str        # explicit | inferred
    evidence: str      # the text that supports this skill
    years: float = 0.0
    proficiency: str = "intermediate"

@dataclass
class ResumeProfile:
    """Structured candidate profile from resume parsing."""
    name: str
    skills: List[ExtractedSkill] = field(default_factory=list)
    total_years: float = 0.0
    education: str = ""
    achievements: List[str] = field(default_factory=list)
    domains: List[str] = field(default_factory=list)

RESUME_PARSE_PROMPT = """You are an expert technical recruiter. Parse this resume
into a structured candidate profile.

RESUME:
{resume_text}

Extract:
1. All skills (both explicitly mentioned AND inferred from experience)
2. For inferred skills, cite the evidence text
3. Total years of experience
4. Education details
5. Key achievements (quantified where possible)

Respond in this exact JSON format:
{{
  "name": "<candidate name>",
  "total_years": <float>,
  "education": "<highest degree and field>",
  "skills": [
    {{"name": "<skill>", "source": "explicit|inferred", "evidence": "<text>", "years": <float>, "proficiency": "<level>"}},
    ...
  ],
  "achievements": ["<achievement1>", ...],
  "domains": ["<domain1>", ...]
}}"""

def parse_resume(resume_text: str) -> ResumeProfile:
    """Parse a resume into a structured candidate profile."""
    if client:
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": RESUME_PARSE_PROMPT.format(
                resume_text=resume_text)}],
            temperature=0.0,
            response_format={"type": "json_object"},
        )
        data = json.loads(resp.choices[0].message.content)
    else:
        # ── Mock: keyword-based extraction ──
        skills: List[Dict] = []
        known = ["python", "java", "sql", "pytorch", "tensorflow", "kubernetes",
                 "docker", "aws", "machine learning", "data analysis", "react",
                 "node.js", "r", "statistics", "deep learning"]
        text_lower = resume_text.lower()
        for skill in known:
            if skill in text_lower:
                skills.append({"name": skill.title(), "source": "explicit",
                               "evidence": f"Found '{skill}' in resume",
                               "years": 2.0, "proficiency": "intermediate"})
        # Infer skills from context
        inferences: Dict[str, str] = {
            "microservices": "Distributed Systems",
            "deployed": "DevOps",
            "pipeline": "Data Engineering",
            "led": "Leadership",
            "published": "Research",
        }
        for kw, inferred in inferences.items():
            if kw in text_lower:
                skills.append({"name": inferred, "source": "inferred",
                               "evidence": f"Context: '{kw}' in resume",
                               "years": 0.0, "proficiency": "intermediate"})
        data = {"name": "Candidate", "total_years": 5.0,
                "education": "BS Computer Science",
                "skills": skills, "achievements": [], "domains": []}

    return ResumeProfile(
        name=data.get("name", "Unknown"),
        total_years=data.get("total_years", 0.0),
        education=data.get("education", ""),
        skills=[ExtractedSkill(**s) for s in data.get("skills", [])],
        achievements=data.get("achievements", []),
        domains=data.get("domains", []),
    )

# ── Test ──
sample_resume = """Dr. Sarah Kim — Machine Learning Researcher
8 years of experience in applied ML and NLP.

Experience:
- Lead ML Engineer at TechCorp (2020-present): Built and deployed
  production recommendation systems serving 10M users. Led team of 4.
  Reduced inference latency by 60% using model distillation.
- Research Scientist at AI Lab (2017-2020): Published 5 papers on
  transformer architectures. Developed novel attention mechanisms.
- Data Scientist at StartupX (2015-2017): Built data pipelines
  processing 1TB/day. Python, SQL, Spark.

Skills: Python, PyTorch, TensorFlow, SQL, Kubernetes, Docker, MLflow
Education: PhD Computer Science, MIT"""

profile = parse_resume(sample_resume)
print("=" * 60)
print(f"  PARSED RESUME: {profile.name}")
print(f"  Experience: {profile.total_years} years | Education: {profile.education}")
print("=" * 60)
explicit = [s for s in profile.skills if s.source == "explicit"]
inferred = [s for s in profile.skills if s.source == "inferred"]
print(f"  Explicit skills ({len(explicit)}):")
for s in explicit:
    print(f"    • {s.name} ({s.proficiency})")
print(f"  Inferred skills ({len(inferred)}):")
for s in inferred:
    print(f"    • {s.name} — from: {s.evidence[:50]}")
print(f"\n✓ Extracted {len(profile.skills)} skills "
      f"({len(explicit)} explicit, {len(inferred)} inferred)")

## Section 4 — Semantic matching

For each job requirement, the semantic matcher searches the candidate profile for
evidence. It classifies each match as:

| Level | Description | Score |
|-------|-------------|-------|
| **Exact** | Skill name directly matches | 1.0 |
| **Semantic** | High embedding similarity (≥ 0.7) | 0.7–0.9 |
| **Partial** | Moderate similarity (0.4–0.7) | 0.4–0.6 |
| **None** | No meaningful match found | 0.0 |

In [ ]:
@dataclass
class MatchResult:
    """Result of matching a single requirement against a candidate."""
    requirement_id: str
    requirement_text: str
    match_level: str      # exact | semantic | partial | none
    score: float          # 0.0 – 1.0
    evidence: str         # what in the resume supports this match
    candidate_skill: str  # the matched skill name

def semantic_match(
    requirements: List[Requirement],
    profile: ResumeProfile,
    threshold_semantic: float = 0.70,
    threshold_partial: float = 0.40,
) -> List[MatchResult]:
    """Match each requirement against candidate skills using embeddings."""
    results: List[MatchResult] = []
    if not profile.skills:
        return [MatchResult(r.id, r.text, "none", 0.0, "", "")
                for r in requirements]

    req_texts = [r.text for r in requirements]
    skill_texts = [s.name for s in profile.skills]

    req_embs = encoder.encode(req_texts)
    skill_embs = encoder.encode(skill_texts)
    sim = cosine_similarity(req_embs, skill_embs)

    for i, req in enumerate(requirements):
        best_j = int(np.argmax(sim[i]))
        best_score = float(sim[i][best_j])
        best_skill = profile.skills[best_j]

        # Check exact match first
        if req.text.lower().strip() in best_skill.name.lower() or            best_skill.name.lower() in req.text.lower():
            level = "exact"
            score = 1.0
        elif best_score >= threshold_semantic:
            level = "semantic"
            score = best_score
        elif best_score >= threshold_partial:
            level = "partial"
            score = best_score
        else:
            level = "none"
            score = 0.0

        results.append(MatchResult(
            requirement_id=req.id,
            requirement_text=req.text,
            match_level=level,
            score=round(score, 3),
            evidence=best_skill.evidence,
            candidate_skill=best_skill.name,
        ))
    return results

# ── Test matcher ──
matches = semantic_match(parsed.requirements, profile)
print("=" * 60)
print("  SEMANTIC MATCHING RESULTS")
print("=" * 60)
level_colors = {"exact": "✓✓", "semantic": "✓ ", "partial": "~ ", "none": "✗ "}
for m in matches:
    tag = level_colors.get(m.match_level, "  ")
    print(f"  {tag} {m.requirement_id}: {m.requirement_text[:40]}")
    print(f"       → {m.match_level} ({m.score:.2f}) — {m.candidate_skill}")
coverage = sum(1 for m in matches if m.match_level != "none") / len(matches)
print(f"\n  Coverage: {coverage:.0%} of requirements matched")
print(f"✓ {sum(1 for m in matches if m.match_level == 'exact')} exact, "
      f"{sum(1 for m in matches if m.match_level == 'semantic')} semantic, "
      f"{sum(1 for m in matches if m.match_level == 'partial')} partial, "
      f"{sum(1 for m in matches if m.match_level == 'none')} none")

## Section 5 — Structured rubric

The rubric evaluator takes match results and scores the candidate across
weighted criteria. Each criterion has anchored scales (1–5) to ensure consistent
evaluation across candidates.

In [ ]:
@dataclass
class CriterionScore:
    """Score for one rubric criterion."""
    criterion: str
    weight: float
    score: int        # 1–5
    evidence: str
    reasoning: str

@dataclass
class Scorecard:
    """Complete evaluation scorecard for a candidate."""
    candidate_name: str
    job_title: str
    criterion_scores: List[CriterionScore] = field(default_factory=list)
    overall_score: float = 0.0
    recommendation: str = ""

def evaluate_candidate(
    profile: ResumeProfile,
    matches: List[MatchResult],
    job_title: str,
) -> Scorecard:
    """Evaluate candidate using structured rubric."""
    # ── Technical skills score ──
    tech_matches = [m for m in matches
                    if m.match_level in ("exact", "semantic")]
    tech_score = min(5, max(1, int(len(tech_matches) / max(len(matches), 1) * 5) + 1))

    # ── Experience score ──
    yrs = profile.total_years
    exp_score = (1 if yrs < 1 else 2 if yrs < 3 else 3 if yrs < 5
                 else 4 if yrs < 8 else 5)

    # ── Education score ──
    edu_lower = profile.education.lower()
    edu_score = (5 if "phd" in edu_lower else 4 if "master" in edu_lower or "ms" in edu_lower
                 else 3 if "bachelor" in edu_lower or "bs" in edu_lower else 2)

    # ── Culture/communication (heuristic from achievements) ──
    culture_score = min(5, max(2, len(profile.achievements)))

    # ── Growth potential ──
    growth_score = min(5, max(2, len(profile.domains) + 1))

    criteria = [
        CriterionScore("Technical Skills", 0.35, tech_score,
                       f"{len(tech_matches)}/{len(matches)} requirements matched",
                       "Based on semantic matching results"),
        CriterionScore("Experience Depth", 0.25, exp_score,
                       f"{profile.total_years} years total experience",
                       f"Score mapped from years: {yrs}"),
        CriterionScore("Education", 0.15, edu_score,
                       profile.education,
                       "Mapped from highest degree"),
        CriterionScore("Culture & Communication", 0.15, culture_score,
                       f"{len(profile.achievements)} achievements listed",
                       "Inferred from achievements and domains"),
        CriterionScore("Growth Potential", 0.10, growth_score,
                       f"{len(profile.domains)} domains",
                       "Breadth of experience"),
    ]

    overall = sum(c.weight * c.score for c in criteria)
    rec = ("strong-yes" if overall >= 4.0 else "yes" if overall >= 3.2
           else "maybe" if overall >= 2.5 else "no")

    return Scorecard(
        candidate_name=profile.name,
        job_title=job_title,
        criterion_scores=criteria,
        overall_score=round(overall, 2),
        recommendation=rec,
    )

# ── Test evaluator ──
scorecard = evaluate_candidate(profile, matches, parsed.title)

print("╔" + "═" * 58 + "╗")
print(f"║  SCORECARD: {scorecard.candidate_name:<44}║")
print(f"║  Position : {scorecard.job_title:<44}║")
print(f"║  Overall  : {scorecard.overall_score:.1f}/5.0 → {scorecard.recommendation:<31}║")
print("╠" + "═" * 58 + "╣")
for cs in scorecard.criterion_scores:
    bar = "█" * cs.score + "░" * (5 - cs.score)
    print(f"║  {cs.criterion:<24} {bar} {cs.score}/5  w={cs.weight:.0%}{' ' * 7}║")
    print(f"║    {cs.evidence[:52]:<56}║")
print("╚" + "═" * 58 + "╝")

## Section 6 — Bias detection

Bias detection operates at two levels:

1. **Score distribution analysis** — Do scores differ systematically across
   demographic groups (simulated)? Calculate adverse impact ratio (4/5ths rule).
2. **JD language analysis** — Does the job description use gendered or exclusionary
   language that discourages diverse applicants?

In [ ]:
@dataclass
class FairnessMetric:
    """A single fairness measurement."""
    metric_name: str
    group_a: str
    group_b: str
    value_a: float
    value_b: float
    ratio: float
    passes_threshold: bool  # True if ratio >= 0.8 (4/5ths rule)

def compute_adverse_impact(
    scores_a: List[float],
    scores_b: List[float],
    threshold: float = 3.0,
) -> FairnessMetric:
    """Compute adverse impact ratio between two groups."""
    rate_a = sum(1 for s in scores_a if s >= threshold) / len(scores_a) if scores_a else 0
    rate_b = sum(1 for s in scores_b if s >= threshold) / len(scores_b) if scores_b else 0
    ratio = rate_b / rate_a if rate_a > 0 else 1.0
    return FairnessMetric(
        metric_name="Adverse Impact Ratio",
        group_a="Group A", group_b="Group B",
        value_a=round(rate_a, 3), value_b=round(rate_b, 3),
        ratio=round(ratio, 3),
        passes_threshold=ratio >= 0.8,
    )

# ── Gendered language detector ──
GENDERED_TERMS: Dict[str, str] = {
    "aggressive": "masculine-coded", "dominant": "masculine-coded",
    "ninja": "masculine-coded", "rockstar": "masculine-coded",
    "competitive": "masculine-coded", "crush it": "masculine-coded",
    "nurturing": "feminine-coded", "supportive": "feminine-coded",
    "collaborative": "neutral", "inclusive": "neutral",
}

def check_jd_language(jd_text: str) -> Dict[str, List[str]]:
    """Check JD for gendered or exclusionary language."""
    findings: Dict[str, List[str]] = defaultdict(list)
    text_lower = jd_text.lower()
    for term, category in GENDERED_TERMS.items():
        if term in text_lower:
            findings[category].append(term)
    return dict(findings)

# ── Simulate score distributions (two demographic groups) ──
np.random.seed(42)
group_a_scores = np.clip(np.random.normal(3.5, 0.8, 50), 1, 5).tolist()
group_b_scores = np.clip(np.random.normal(3.2, 0.9, 50), 1, 5).tolist()

ai_metric = compute_adverse_impact(group_a_scores, group_b_scores)

# ── Test JD language ──
biased_jd = ("We need a rockstar engineer who can crush it in a competitive, "
             "fast-paced, aggressive environment. Must be a coding ninja.")
neutral_jd = ("We are looking for a collaborative engineer who thrives in an "
              "inclusive, supportive team environment.")

biased_findings = check_jd_language(biased_jd)
neutral_findings = check_jd_language(neutral_jd)

print("=" * 60)
print("  BIAS DETECTION — SCORE DISTRIBUTION")
print("=" * 60)
print(f"  Group A pass rate (≥3.0): {ai_metric.value_a:.1%}")
print(f"  Group B pass rate (≥3.0): {ai_metric.value_b:.1%}")
print(f"  Adverse impact ratio    : {ai_metric.ratio:.3f}")
print(f"  4/5ths rule             : {'✓ PASS' if ai_metric.passes_threshold else '✗ FAIL'}")

print("\n" + "-" * 60)
print("  JD LANGUAGE ANALYSIS")
print("-" * 60)
print("  Biased JD findings:")
for cat, terms in biased_findings.items():
    print(f"    {cat}: {', '.join(terms)}")
print("  Neutral JD findings:")
if neutral_findings:
    for cat, terms in neutral_findings.items():
        print(f"    {cat}: {', '.join(terms)}")
else:
    print("    ✓ No gendered language detected")
print("\n✓ Bias detection catches both score disparities and language issues.")

In [ ]:
# ── Visualise match level distribution and scorecard ──
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Match level distribution
levels = ["exact", "semantic", "partial", "none"]
counts = [sum(1 for m in matches if m.match_level == lv) for lv in levels]
colors = ["#388e3c", "#1976d2", "#f57c00", "#d32f2f"]
axes[0].bar(levels, counts, color=colors)
axes[0].set_ylabel("Count")
axes[0].set_title("Match level distribution")

# Scorecard criteria bar chart
criteria_names = [cs.criterion for cs in scorecard.criterion_scores]
criteria_scores = [cs.score for cs in scorecard.criterion_scores]
criteria_weights = [cs.weight for cs in scorecard.criterion_scores]
weighted = [s * w for s, w in zip(criteria_scores, criteria_weights)]
axes[1].barh(criteria_names, criteria_scores, color="#1976d2", alpha=0.5, label="Raw")
axes[1].barh(criteria_names, weighted, color="#1976d2", label="Weighted")
axes[1].set_xlim(0, 5.5)
axes[1].set_xlabel("Score")
axes[1].set_title(f"Scorecard: {scorecard.candidate_name}")
axes[1].legend()

plt.tight_layout()
plt.show()
print("✓ Architecture visualisations rendered")

## Exercises

1. **Enhance the JD parser** — Modify `parse_jd` to also extract *team size*,
   *reporting structure*, and *remote/onsite* from the job description. Add these
   fields to `ParsedJD` and test with 3 different JDs.

2. **Skill inference rules** — Build a mapping of 20 experience phrases to
   inferred skills (e.g., "built CI/CD pipelines" → DevOps, Jenkins, GitHub
   Actions). Add these to the resume parser and measure how many additional
   skills are captured.

3. **Fairness calibration** — Implement a score calibration function that adjusts
   raw scores to equalise pass rates across groups while minimising total score
   distortion. Test with simulated data.

## Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | The pipeline has 6 stages: parse JD → parse resume → match → evaluate → check bias → generate questions |
| 2 | JD parsing must separate required vs preferred requirements for accurate matching |
| 3 | Resume parsing should infer skills from context, not just extract explicit mentions |
| 4 | Semantic matching captures exact, semantic, partial, and no-match levels |
| 5 | Structured rubrics with anchored scales produce consistent, explainable scores |
| 6 | Bias detection operates at both score-distribution and language levels |
| 7 | The 4/5ths rule provides a clear, legally-grounded threshold for adverse impact |

## What's Next

In **02 — Build**, we implement the full recruitment matching engine with
realistic job descriptions, diverse candidate resumes, and end-to-end scoring
with fairness analysis.

---

## References

1. EEOC, *Uniform Guidelines on Employee Selection Procedures (4/5ths rule)*, 1978 — <https://www.eeoc.gov/>
2. Gaucher, D. et al., *Evidence That Gendered Wording in Job Advertisements Exists and Sustains Gender Inequality*, Journal of Personality and Social Psychology, 2011.
3. Sentence-Transformers library — <https://www.sbert.net/>
4. NYC Local Law 144 — *Automated Employment Decision Tools*, 2023 — <https://www.nyc.gov/>